# HQ200: structure and example readers

This notebook inventories the five ARKit capture scenes and reads representative RGB, depth, confidence, camera-pose, mesh, material, annotation, and world-map files. It does not modify the dataset.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


In [ ]:
# Override this line if the project is moved.
PROJECT_ROOT = Path(r"C:\Users\liter\Documents\ITU_MSc\Project_Thesis")
HQ200_ROOT = PROJECT_ROOT / "data" / "HQ200"

if not HQ200_ROOT.is_dir():
    # Also support a kernel whose working directory is the project or ResearchProject folder.
    candidates = [Path.cwd() / "data" / "HQ200", Path.cwd().parent / "data" / "HQ200"]
    HQ200_ROOT = next((p.resolve() for p in candidates if p.is_dir()), HQ200_ROOT)

if not HQ200_ROOT.is_dir():
    raise FileNotFoundError(f"HQ200 directory not found: {HQ200_ROOT}")

SCENES = sorted(p for p in HQ200_ROOT.iterdir() if p.is_dir())
print(f"HQ200 root: {HQ200_ROOT}")
print(f"Scenes: {len(SCENES)}")


## 1. Dataset inventory

In [ ]:
def scene_inventory(scene):
    files = [p for p in scene.iterdir() if p.is_file()]
    return {
        "scene": scene.name,
        "rgb_frames": len(list(scene.glob("frame_*.jpg"))),
        "camera_json": len(list(scene.glob("frame_*.json"))),
        "depth_maps": len(list(scene.glob("depth_*.png"))),
        "confidence_maps": len(list(scene.glob("conf_*.png"))),
        "obj_meshes": len(list(scene.glob("*.obj"))),
        "arkit_maps": len(list(scene.glob("*.arkit"))),
        "files_total": len(files),
        "size_mb": round(sum(p.stat().st_size for p in files) / 1024**2, 1),
    }

inventory = pd.DataFrame(scene_inventory(scene) for scene in SCENES)
display(inventory)
print(f"Total size: {inventory['size_mb'].sum() / 1024:.3f} GB")


## 2. Select a scene and aligned frame

In [ ]:
SCENE_INDEX = 0
FRAME_INDEX = 0

scene = SCENES[SCENE_INDEX]
stem = f"{FRAME_INDEX:05d}"
rgb_path = scene / f"frame_{stem}.jpg"
depth_path = scene / f"depth_{stem}.png"
confidence_path = scene / f"conf_{stem}.png"
camera_path = scene / f"frame_{stem}.json"

print("Scene:", scene.name)
for label, path in {"RGB": rgb_path, "depth": depth_path, "confidence": confidence_path, "camera": camera_path}.items():
    print(f"{label:10s} {path.name:22s} exists={path.is_file()}")

# RGB images are sampled less frequently than depth/camera frames.
if not rgb_path.is_file():
    rgb_path = min(scene.glob("frame_*.jpg"), key=lambda p: abs(int(p.stem.split('_')[1]) - FRAME_INDEX))
    print("Nearest available RGB frame:", rgb_path.name)


In [ ]:
rgb = np.asarray(Image.open(rgb_path).convert("RGB"))
depth_raw = np.asarray(Image.open(depth_path))
confidence = np.asarray(Image.open(confidence_path))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(rgb)
axes[0].set_title(f"RGB: {rgb_path.name}\n{rgb.shape[1]} x {rgb.shape[0]}")
depth_view = axes[1].imshow(depth_raw, cmap="turbo")
axes[1].set_title(f"Depth (raw uint16): {depth_path.name}\nrange {depth_raw.min()}-{depth_raw.max()}")
fig.colorbar(depth_view, ax=axes[1], fraction=0.046)
conf_view = axes[2].imshow(confidence, cmap="viridis", vmin=0, vmax=2)
axes[2].set_title(f"Confidence: {confidence_path.name}\nvalues {np.unique(confidence).tolist()}")
fig.colorbar(conf_view, ax=axes[2], ticks=[0, 1, 2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Camera metadata

In [ ]:
camera = json.loads(camera_path.read_text(encoding="utf-8"))
intrinsics = np.asarray(camera["intrinsics"], dtype=float).reshape(3, 3)
camera_pose = np.asarray(camera["cameraPoseARFrame"], dtype=float).reshape(4, 4)
projection = np.asarray(camera["projectionMatrix"], dtype=float).reshape(4, 4)

print("Available fields:", sorted(camera))
print("Frame index:", camera.get("frame_index"))
print("Time:", camera.get("time"))
print("Motion quality:", camera.get("motionQuality"))
print("Intrinsics:\n", intrinsics)
print("Camera pose (AR frame):\n", camera_pose)
print("Projection matrix:\n", projection)


In [ ]:
# Visualize the complete ARKit camera trajectory and a subset of camera frustums.
# cameraPoseARFrame is treated as camera-to-world; ARKit cameras look along local -Z.
camera_files = sorted(scene.glob("frame_*.json"))
poses = []
frame_ids = []
for path in camera_files:
    metadata = json.loads(path.read_text(encoding="utf-8"))
    poses.append(np.asarray(metadata["cameraPoseARFrame"], dtype=float).reshape(4, 4))
    frame_ids.append(int(metadata.get("frame_index", path.stem.split("_")[-1])))
poses = np.stack(poses)
centers = poses[:, :3, 3]

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")

# Optional raw-mesh overlay. OBJ vertices form a point set as well as mesh vertices.
mesh_path = scene / "export.obj"
mesh_vertices = []
with mesh_path.open(encoding="utf-8", errors="replace") as handle:
    for line in handle:
        if line.startswith("v "):
            mesh_vertices.append([float(value) for value in line.split()[1:4]])
mesh_vertices = np.asarray(mesh_vertices)
mesh_step = max(1, len(mesh_vertices) // 10000)
ax.scatter(*mesh_vertices[::mesh_step].T, s=0.4, c="lightgray", alpha=0.25, label="export.obj vertices")

trajectory = ax.scatter(*centers.T, c=frame_ids, cmap="viridis", s=6, label="camera centers")
ax.plot(*centers.T, color="black", linewidth=0.5, alpha=0.5)

# Draw about 30 frustums. R columns are camera right, up, and backward axes.
frustum_step = max(1, len(poses) // 30)
frustum_depth = max(np.ptp(centers, axis=0).max() * 0.035, 0.08)
for transform in poses[::frustum_step]:
    rotation = transform[:3, :3]
    center = transform[:3, 3]
    right, up, backward = rotation[:, 0], rotation[:, 1], rotation[:, 2]
    forward = -backward
    target = center + frustum_depth * forward
    half_width = 0.55 * frustum_depth
    half_height = 0.40 * frustum_depth
    corners = np.array([
        target - half_width * right - half_height * up,
        target + half_width * right - half_height * up,
        target + half_width * right + half_height * up,
        target - half_width * right + half_height * up,
    ])
    for corner in corners:
        ax.plot(*np.vstack([center, corner]).T, color="tab:red", linewidth=0.7, alpha=0.7)
    closed = np.vstack([corners, corners[0]])
    ax.plot(*closed.T, color="tab:red", linewidth=0.7, alpha=0.7)

# Equal scaling prevents a distorted trajectory.
all_points = np.vstack([centers, mesh_vertices])
lower, upper = all_points.min(axis=0), all_points.max(axis=0)
mid = (lower + upper) / 2
radius = (upper - lower).max() / 2
ax.set_xlim(mid[0] - radius, mid[0] + radius)
ax.set_ylim(mid[1] - radius, mid[1] + radius)
ax.set_zlim(mid[2] - radius, mid[2] + radius)
ax.set_xlabel("ARKit X")
ax.set_ylabel("ARKit Y")
ax.set_zlabel("ARKit Z")
ax.set_title(f"ARKit camera poses: {scene.name}")
fig.colorbar(trajectory, ax=ax, shrink=0.65, label="frame index")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 4. Mesh, material, annotations, and ARKit world map

In [ ]:
def summarize_obj(path):
    counts = Counter()
    with path.open(encoding="utf-8", errors="replace") as handle:
        for line in handle:
            token = line.split(maxsplit=1)[0] if line.strip() else "blank"
            counts[token] += 1
    return {
        "file": path.name,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "vertices": counts["v"],
        "texture_coords": counts["vt"],
        "normals": counts["vn"],
        "faces": counts["f"],
    }

mesh_summary = pd.DataFrame(summarize_obj(path) for path in sorted(scene.glob("*.obj")))
display(mesh_summary)

annotations_path = scene / "annotations.json"
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))
print("Annotations:", annotations)

for material_path in sorted(scene.glob("*.mtl")):
    print(f"\nMaterial file: {material_path.name}")
    print("".join(material_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[:20]))

for world_map_path in sorted(scene.glob("*.arkit")):
    header = world_map_path.read_bytes()[:8]
    print(f"ARKit map: {world_map_path.name}, {world_map_path.stat().st_size / 1024**2:.2f} MB, header={header!r}")
    print("The bplist00 header identifies an Apple binary property-list archive; this notebook reports it without attempting unsafe object deserialization.")


## 5. Browse several RGB examples

In [ ]:
N_EXAMPLES = 8
rgb_paths = sorted(scene.glob("frame_*.jpg"))
sample_positions = np.linspace(0, len(rgb_paths) - 1, min(N_EXAMPLES, len(rgb_paths)), dtype=int)
sample_paths = [rgb_paths[i] for i in sample_positions]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.flat, sample_paths):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(path.name)
    ax.axis("off")
for ax in axes.flat[len(sample_paths):]:
    ax.axis("off")
plt.suptitle(scene.name)
plt.tight_layout()
plt.show()


## 6. What the geometry files mean

An OBJ is a **triangle mesh**: `v` records are 3D vertices and `f` records connect them into faces. The vertices alone can be treated as a point cloud. `export.obj` is coarse, `export_refined.obj` is denser, and `textured_output.obj` adds UV coordinates that connect it to `textured_output.jpg` through `textured_output.mtl`. Missing stored normals are not an error; they can be calculated from the faces.

`world_map.arkit` is an Apple [`ARWorldMap`](https://developer.apple.com/documentation/arkit/arworldmap) archive used primarily for AR relocalization. Its [`rawFeaturePoints`](https://developer.apple.com/documentation/arkit/arworldmap/rawfeaturepoints) may contain sparse AR tracking points, but it is not a portable replacement for the OBJ mesh or the paper's cleaned point cloud. Full decoding normally requires ARKit on macOS/iOS.

## 7. Interactive mesh and point-cloud exploration

The next cells use only NumPy and Plotly. Large meshes are display-subsampled, while statistics are computed from the complete file.

In [ ]:
def load_obj_geometry(path):
    vertices, texture_coordinates, triangles = [], [], []
    with Path(path).open(encoding="utf-8", errors="replace") as handle:
        for line in handle:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "v":
                vertices.append([float(value) for value in parts[1:4]])
            elif parts[0] == "vt":
                texture_coordinates.append([float(value) for value in parts[1:3]])
            elif parts[0] == "f":
                polygon = [int(item.split("/")[0]) - 1 for item in parts[1:]]
                # Fan triangulation also supports OBJ faces with more than three vertices.
                triangles.extend([polygon[0], polygon[i], polygon[i + 1]] for i in range(1, len(polygon) - 1))
    return np.asarray(vertices, dtype=np.float32), np.asarray(texture_coordinates, dtype=np.float32), np.asarray(triangles, dtype=np.int32)

MESH_NAME = "textured_output.obj"  # also try export.obj or export_refined.obj
selected_mesh_path = scene / MESH_NAME
vertices, uv_coordinates, triangles = load_obj_geometry(selected_mesh_path)
extent = np.ptp(vertices, axis=0)
print("Mesh:", selected_mesh_path.name)
print("Vertices:", len(vertices), "Triangles:", len(triangles), "UV coordinates:", len(uv_coordinates))
print("Bounds min:", vertices.min(axis=0), "max:", vertices.max(axis=0))
print("Extent:", extent, "diagonal:", np.linalg.norm(extent))


In [ ]:
import plotly.graph_objects as go

MAX_DISPLAY_FACES = 60_000
face_step = max(1, int(np.ceil(len(triangles) / MAX_DISPLAY_FACES)))
display_faces = triangles[::face_step]

mesh_figure = go.Figure(go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
    i=display_faces[:, 0], j=display_faces[:, 1], k=display_faces[:, 2],
    color="lightsteelblue", opacity=1.0, flatshading=False,
))
mesh_figure.update_layout(
    title=f"{selected_mesh_path.name}: {len(display_faces):,} displayed triangles",
    scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    height=750, margin=dict(l=0, r=0, b=0, t=45),
)
mesh_figure.show()


In [ ]:
texture_path = scene / "textured_output.jpg"
if texture_path.is_file():
    texture = Image.open(texture_path).convert("RGB")
    plt.figure(figsize=(14, 7))
    plt.imshow(texture)
    plt.title(f"Texture atlas: {texture_path.name} ({texture.width} x {texture.height})")
    plt.axis("off")
    plt.show()
else:
    print("No textured_output.jpg found.")


In [ ]:
def sample_mesh_surface(vertices, triangles, n_points=100_000, seed=42):
    rng = np.random.default_rng(seed)
    tri = vertices[triangles]
    areas = np.linalg.norm(np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1) / 2
    valid = areas > 0
    tri, areas = tri[valid], areas[valid]
    chosen = rng.choice(len(tri), size=n_points, p=areas / areas.sum())
    selected = tri[chosen]
    u, v = rng.random((2, n_points))
    swap = u + v > 1
    u[swap], v[swap] = 1 - u[swap], 1 - v[swap]
    return selected[:, 0] + u[:, None] * (selected[:, 1] - selected[:, 0]) + v[:, None] * (selected[:, 2] - selected[:, 0])

surface_points = sample_mesh_surface(vertices, triangles)
display_step = max(1, len(surface_points) // 25_000)
shown_points = surface_points[::display_step]
point_figure = go.Figure(go.Scatter3d(
    x=shown_points[:, 0], y=shown_points[:, 1], z=shown_points[:, 2],
    mode="markers", marker=dict(size=1, color=shown_points[:, 2], colorscale="Viridis"),
))
point_figure.update_layout(title="Uniformly sampled mesh-surface point cloud", scene_aspectmode="data", height=700)
point_figure.show()


In [ ]:
EXPORT_POINT_CLOUD = False
POINT_CLOUD_PATH = PROJECT_ROOT / "outputs" / "HQ200" / scene.name / "mesh_surface_points.ply"

def save_ascii_ply(path, points):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="ascii") as handle:
        handle.write("ply\nformat ascii 1.0\n")
        handle.write(f"element vertex {len(points)}\n")
        handle.write("property float x\nproperty float y\nproperty float z\nend_header\n")
        np.savetxt(handle, points, fmt="%.6f %.6f %.6f")

if EXPORT_POINT_CLOUD:
    save_ascii_ply(POINT_CLOUD_PATH, surface_points)
    print("Saved:", POINT_CLOUD_PATH)
else:
    print("Set EXPORT_POINT_CLOUD=True to write a PLY file.")


## 8. PCA orientation preview

The paper standardizes each cleaned car point cloud with PCA and then manually checks its facing direction. This preview applies PCA to the selected raw mesh only; it is diagnostic, not a substitute for foreground cleaning or manual validation.

In [ ]:
center = surface_points.mean(axis=0)
centered_points = surface_points - center
eigenvalues, axes_pca = np.linalg.eigh(np.cov(centered_points, rowvar=False))
order = np.argsort(eigenvalues)[::-1]
axes_pca = axes_pca[:, order]
if np.linalg.det(axes_pca) < 0:
    axes_pca[:, -1] *= -1
standardized_points = centered_points @ axes_pca
print("PCA eigenvalues:", eigenvalues[order])
print("Standardized extent:", np.ptp(standardized_points, axis=0))

shown = standardized_points[::display_step]
pca_figure = go.Figure(go.Scatter3d(
    x=shown[:, 0], y=shown[:, 1], z=shown[:, 2], mode="markers",
    marker=dict(size=1, color=shown[:, 0], colorscale="Turbo"),
))
pca_figure.update_layout(title="PCA-standardized preview (orientation still needs checking)", scene_aspectmode="data", height=700)
pca_figure.show()


## 9. Paper-faithful reconstruction preprocessing

The [3DRealCar paper](https://arxiv.org/html/2406.04875v1) reports that scanner poses were not accurate enough for its final reconstructions. Its released [preprocessing toolkit](https://github.com/xiaobiaodu/3DRealCar_Toolkit/tree/master/data_preprocess) runs these stages:

1. **COLMAP:** estimate refined intrinsics, camera poses, and a sparse point cloud.
2. **Segmentation:** Grounding DINO locates the car and SAM produces foreground masks.
3. **Point-cloud cleaning:** project points into the masks and remove background geometry.
4. **Standardization:** use PCA to align the car; manual checking is required.
5. **Rescaling:** align COLMAP scale with the real-scale ARKit scanner trajectory.
6. **Reconstruction:** use the processed images, poses, masks, and point cloud with 3DGS, 2DGS, GaussianShader, or Instant-NGP.

> The official scripts are Linux-oriented and computationally heavy. Run the controlled cells below in a hosted Colab GPU runtime. Raw HQ200 files are linked or copied into a separate workspace and are never moved or overwritten.

In [ ]:
import os
import platform
import shutil
import subprocess

RUN_TOOLKIT_SETUP = False
RUN_PIPELINE_STAGES = []  # add in order: dataset, segmentation, pcd_clean, pcd_standard, pcd_rescale
EXPERIMENT_NAME = "demo"
RECON_ROOT = PROJECT_ROOT / "outputs" / "HQ200_reconstruction"
TOOLKIT_ROOT = RECON_ROOT / "3DRealCar_Toolkit"
RAW_DATA_ROOT = RECON_ROOT / "raw_data"
PIPELINE_DATASET_NAME = scene.name.replace(" ", "_")  # official bash scripts do not safely quote spaces
PIPELINE_SCENE_ROOT = RAW_DATA_ROOT / PIPELINE_DATASET_NAME
SCANNER_ORIGIN = PIPELINE_SCENE_ROOT / "3dscanner_origin"

print("Platform:", platform.system())
print("Selected scene:", scene)
print("Reconstruction workspace:", RECON_ROOT)
print("Stages requested:", RUN_PIPELINE_STAGES or "none")


In [ ]:
def run_checked(command, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=True, text=True)

if RUN_TOOLKIT_SETUP:
    if platform.system() != "Linux":
        raise RuntimeError("The released 3DRealCar pipeline uses bash, apt, and Linux symlinks. Run this section in Colab.")
    RECON_ROOT.mkdir(parents=True, exist_ok=True)
    if not TOOLKIT_ROOT.is_dir():
        run_checked(["git", "clone", "--recursive", "https://github.com/xiaobiaodu/3DRealCar_Toolkit.git", str(TOOLKIT_ROOT)])
    run_checked(["apt-get", "update"])
    run_checked(["apt-get", "install", "-y", "colmap"])
    run_checked(["python", "-m", "pip", "install", "colorama", "plyfile", "open3d", "kornia", "tqdm", "imageio[ffmpeg]", "opencv-python", "supervision", "segment_anything"])
    grounding_dino = TOOLKIT_ROOT / "data_preprocess" / "submodules" / "GroundingDINO"
    if grounding_dino.is_dir():
        run_checked(["python", "-m", "pip", "install", str(grounding_dino)])
    model_dir = TOOLKIT_ROOT / "data_preprocess" / "resources" / "models"
    model_dir.mkdir(parents=True, exist_ok=True)
    sam_checkpoint = model_dir / "sam_vit_h_4b8939.pth"
    if not sam_checkpoint.is_file():
        run_checked(["wget", "-O", str(sam_checkpoint), "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"])
    print("Toolkit setup complete.")
else:
    print("Setup skipped. Set RUN_TOOLKIT_SETUP=True in a Linux/Colab runtime to install the official dependencies.")


In [ ]:
def prepare_raw_scene(source_scene, scanner_origin):
    scanner_origin.parent.mkdir(parents=True, exist_ok=True)
    if scanner_origin.exists() or scanner_origin.is_symlink():
        return
    try:
        scanner_origin.symlink_to(source_scene.resolve(), target_is_directory=True)
        print("Linked raw scene:", scanner_origin, "->", source_scene.resolve())
    except OSError:
        scanner_origin.mkdir(parents=True, exist_ok=True)
        for source in source_scene.iterdir():
            if source.is_file():
                shutil.copy2(source, scanner_origin / source.name)
        print("Symlink unavailable; copied raw files to:", scanner_origin)

if RUN_PIPELINE_STAGES:
    prepare_raw_scene(scene, SCANNER_ORIGIN)
else:
    print("Workspace preparation skipped because no stages were requested.")


In [ ]:
VALID_STAGES = ["dataset", "segmentation", "pcd_clean", "pcd_standard", "pcd_rescale"]
if any(stage not in VALID_STAGES for stage in RUN_PIPELINE_STAGES):
    raise ValueError(f"Stages must be selected from {VALID_STAGES}")
if RUN_PIPELINE_STAGES != VALID_STAGES[:len(RUN_PIPELINE_STAGES)]:
    raise ValueError("Run stages from the beginning and in order because each stage depends on the previous one.")

def make_local_pipeline(toolkit_root, dataset_root):
    preprocess_root = toolkit_root / "data_preprocess"
    official_script = preprocess_root / "bash" / "pipeline.sh"
    local_script = preprocess_root / "bash" / "pipeline_local.sh"
    script = official_script.read_text(encoding="utf-8")
    script = script.replace("dataset_dir=/path/to/save/path", f"dataset_dir={dataset_root}")
    script = script.replace("codebase_dir=/path/to/3DRealCar_Dataset/data_preprocess", f"codebase_dir={preprocess_root}")
    local_script.write_text(script, encoding="utf-8")
    local_script.chmod(0o755)
    return local_script

if RUN_PIPELINE_STAGES:
    if platform.system() != "Linux":
        raise RuntimeError("Run the official stages in a Linux/Colab runtime.")
    if not TOOLKIT_ROOT.is_dir():
        raise FileNotFoundError("Toolkit is missing. Run the setup cell first.")
    pipeline_script = make_local_pipeline(TOOLKIT_ROOT, RAW_DATA_ROOT)
    for requested_stage in RUN_PIPELINE_STAGES:
        run_checked(["bash", str(pipeline_script), PIPELINE_DATASET_NAME, requested_stage, EXPERIMENT_NAME])
else:
    print("No heavy stage executed. Example: RUN_PIPELINE_STAGES=['dataset'] runs COLMAP only.")


### Expected outputs and reconstruction hand-off

After all five stages, inspect `raw_data/<scene>/colmap_processed/pcd_rescale/`. It should contain the processed images/masks, COLMAP cameras, rescaled poses, and `points3D.ply` used to initialize reconstruction. The paper benchmarks Instant-NGP, 3DGS, GaussianShader, and 2DGS. For an eight-view experiment, first establish a full-view reconstruction, then select eight views **after** pose refinement to isolate sparse reconstruction; separately test COLMAP using only eight views to measure sparse pose-estimation failures.